In [0]:
# Check if sentence-transformers is already installed
try:
    import sentence_transformers
    print('✅ sentence-transformers already installed')
except ImportError:
    print('📦 Installing sentence-transformers...')
    %pip install -q sentence-transformers
    print('⚠️  Restarting Python (MLflow will not work after restart)')
    dbutils.library.restartPython()

# Loop de Convergencia - Clasificación Jerárquica Iterativa

## Objetivo
Procesar productos no clasificados en **batches pequeños (100 productos)** con **recálculo de centroids** después de cada batch exitoso.

## Estrategia
1. **Iterar N veces** (ej: 30 iteraciones = 3,000 productos)
2. **Por cada iteración:**
   - Tomar 100 productos random no clasificados
   - Clasificar con thresholds altos:
     - `MACRO_AUTO = 0.90`
     - `CATEGORY_AUTO = 0.75`
   - Si hay productos auto-asignados:
     - Guardar en `gold_productos_categorias`
     - **Ejecutar notebook 05** para recalcular centroids
     - Recargar centroids en memoria
   - Si NO hay productos auto-asignados:
     - Continuar (centroids no cambian)

3. **Tracking de convergencia:**
   - % de requeue por iteración
   - Productos auto-asignados acumulados
   - Ver si mejora con el tiempo

## Hipótesis
A medida que agregamos productos, los centroids se refinan y **deberían capturar mejor la distribución de productos "difíciles"** → el % de requeue debería **disminuir**.

---

In [0]:
import json
import numpy as np
import pandas as pd
from datetime import datetime
from sentence_transformers import SentenceTransformer
import time
import os

# Configure MLflow for serverless - set tracking URI via environment variable to avoid Spark config access
os.environ['MLFLOW_TRACKING_URI'] = 'databricks'
os.environ['MLFLOW_EXPERIMENT_NAME'] = '/Users/franciscobarber@gmail.com/superapp/classification_convergence'

import mlflow
mlflow.autolog(disable=True)  # prevent autologging hooks

print('✅ MLflow configured for serverless')

# Configuration - PRODUCTION RUN: Clean up 18,905 unclassified products
MACRO_AUTO = 0.70      # Threshold para auto-asignar MACRO
CATEGORY_AUTO = 0.70   # Threshold para auto-asignar CATEGORIA
BATCH_SIZE = 100       # Productos por iteracion
MAX_ITERATIONS = 200   # Enough for ~19K products (18,905/100 = ~189 iterations)
REBUILD_CENTROIDS_EVERY = 10  # Recalcular centroids cada 10 iteraciones
MIN_ACCURACY_PCT = 80.0      # Minimo accuracy para promover a gold

# LLM Configuration
USE_LLM = False  # endpoint not configured yet
UNCLASSIFIED_CATEGORY_NAME = 'Sin Clasificar'  # Catch-all category; adjust to match gold_categorias
LLM_MODEL = 'databricks-meta-llama-3-3-70b-instruct'
EMBEDDING_MODEL = 'intfloat/multilingual-e5-small'
LLM_MACRO_MIN = 0.60
LLM_MACRO_MAX = 0.70
LLM_TOP_CANDIDATES = 3

print(f'Configuracion:')
print(f'   MACRO_AUTO:              {MACRO_AUTO}')
print(f'   CATEGORY_AUTO:           {CATEGORY_AUTO}')
print(f'   BATCH_SIZE:              {BATCH_SIZE}')
print(f'   MAX_ITERATIONS:          {MAX_ITERATIONS} (PRODUCTION - clean centroids)')
print(f'   REBUILD_CENTROIDS_EVERY: {REBUILD_CENTROIDS_EVERY}')
print(f'   MIN_ACCURACY_PCT:        {MIN_ACCURACY_PCT}%')
print(f'   USE_LLM:                 {USE_LLM}')
print(f'   Total productos a procesar: {BATCH_SIZE * MAX_ITERATIONS:,}')

In [0]:
def build_macro_prompt(producto, macro_names):
    """Construye prompt para que LLM decida el MACRO"""
    macros_text = ", ".join(macro_names)
    return f"""Eres un clasificador experto de productos de supermercado.

Producto: "{producto}"

Macros disponibles: {macros_text}

¿A qué macro pertenece este producto? Responde SOLO con JSON:
{{
    "macro": "nombre_del_macro",
    "confianza": 0.0-1.0
}}

Si no estás seguro, usa "ninguna" como macro."""

def build_category_prompt(producto, top_candidates, all_categories_dict):
    """Construye prompt para que LLM decida la CATEGORIA"""
    candidates_text = "\n".join([
        f"- {cat} (similitud={sim:.2f})" + 
        (f" ej: {examples[0] if isinstance(examples, list) and examples else examples}" if examples else "")
        for cat, sim, examples in top_candidates
    ])
    
    return f"""Eres un clasificador experto de productos de supermercado.

Producto: "{producto}"

Categorías candidatas:
{candidates_text}

¿A qué categoría pertenece este producto? Responde SOLO con JSON:
{{
    "categoria": "nombre_de_la_categoria",
    "confianza": 0.0-1.0
}}

Si no estás seguro, usa "ninguna" como categoría."""

print('✅ Funciones build_macro_prompt() y build_category_prompt() definidas')

In [0]:
# 🧪 EXPERIMENT ISOLATION FUNCTIONS
# Safe experimentation: staging → validate → promote

def create_experiment_staging_table(run_id):
    """
    Creates a staging table for this experiment run.
    Returns: staging_table_name
    """
    staging_table = f"workspace.superapp.experiment_classifications_{run_id[:8]}"
    
    print(f'📋 Creating staging table: {staging_table}')
    
    # Create table with Delta Lake property to allow column defaults
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {staging_table} (
            id_producto BIGINT,
            id_categoria BIGINT,
            id_subcategoria BIGINT,
            metodo STRING,
            confianza DOUBLE,
            fecha_asignacion TIMESTAMP,
            usuario_asignacion STRING,
            notas STRING,
            run_id STRING,
            iteration INT
        )
        USING DELTA
        TBLPROPERTIES (
            'delta.feature.allowColumnDefaults' = 'supported'
        )
    """)
    
    print(f'✅ Staging table created: {staging_table}')
    return staging_table

def save_to_staging(records, staging_table, run_id, iteration=None):
    """
    Saves classification records to staging table.
    Returns: number of records saved
    """
    if not records:
        return 0
    
    from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType, TimestampType, IntegerType
    
    # Add run_id and iteration to each record
    for record in records:
        record['run_id'] = run_id
        record['iteration'] = iteration
        if 'id_subcategoria' not in record:
            record['id_subcategoria'] = None
    
    # Create pandas DataFrame and explicitly convert types
    pdf = pd.DataFrame(records)
    pdf['id_producto'] = pdf['id_producto'].astype('int64')
    pdf['id_categoria'] = pdf['id_categoria'].astype('int64')
    pdf['id_subcategoria'] = pdf['id_subcategoria'].astype('Int64')
    pdf['confianza'] = pdf['confianza'].astype('float64')
    pdf['fecha_asignacion'] = pd.to_datetime(pdf['fecha_asignacion'])
    pdf['iteration'] = pdf['iteration'].astype('Int32') if iteration is not None else None
    
    # Define schema explicitly
    schema = StructType([
        StructField('id_producto', LongType(), False),
        StructField('id_categoria', LongType(), False),
        StructField('id_subcategoria', LongType(), True),
        StructField('metodo', StringType(), False),
        StructField('confianza', DoubleType(), False),
        StructField('fecha_asignacion', TimestampType(), False),
        StructField('usuario_asignacion', StringType(), False),
        StructField('notas', StringType(), True),
        StructField('run_id', StringType(), False),
        StructField('iteration', IntegerType(), True)
    ])
    
    df = spark.createDataFrame(pdf, schema=schema)
    df.write.mode('append').saveAsTable(staging_table)
    
    print(f'   💾 Saved {len(records)} records to staging')
    return len(records)

def validate_staging_table(staging_table):
    """
    Validates classifications in staging table.
    Returns: dict with validation metrics
    """
    print(f'\n🔍 Validating {staging_table}...')
    
    validation_df = spark.sql(f"""
        SELECT
            COUNT(*) as total_classifications,
            COALESCE(SUM(CASE 
                WHEN (LOWER(sp.producto) LIKE '%vino%' OR LOWER(sp.producto) LIKE '%wine%')
                     AND c.nombre != 'Vinos'
                THEN 1 ELSE 0 
            END), 0) as wine_errors,
            COALESCE(SUM(CASE 
                WHEN (LOWER(sp.producto) LIKE '%cerveza%' OR LOWER(sp.producto) LIKE '%beer%')
                     AND c.nombre != 'Cerveza'
                THEN 1 ELSE 0
            END), 0) as beer_errors,
            COALESCE(SUM(CASE 
                WHEN (LOWER(sp.producto) LIKE '%whisky%' OR LOWER(sp.producto) LIKE '%whiskey%')
                     AND c.nombre != 'Whisky'
                THEN 1 ELSE 0
            END), 0) as whisky_errors,
            COALESCE(SUM(CASE WHEN c.nombre LIKE '_OLD_%' THEN 1 ELSE 0 END), 0) as old_category_errors
        FROM {staging_table} st
        JOIN workspace.superapp.silver_prices sp ON st.id_producto = sp.id_producto
        JOIN workspace.superapp.gold_categorias c ON st.id_categoria = c.id_categoria
    """).collect()[0]
    
    total = validation_df['total_classifications']
    detectable_errors = (
        validation_df['wine_errors'] + 
        validation_df['beer_errors'] + 
        validation_df['whisky_errors'] + 
        validation_df['old_category_errors']
    )
    estimated_accuracy_pct = ((total - detectable_errors) / total * 100) if total > 0 else 0
    
    print(f'\n📊 VALIDATION RESULTS:')
    print(f'   Total classifications: {total:,}')
    print(f'   Detectable errors: {detectable_errors:,}')
    print(f'   Estimated accuracy: {estimated_accuracy_pct:.1f}%')
    
    return {
        'total_classifications': total,
        'detectable_errors': detectable_errors,
        'estimated_accuracy_pct': estimated_accuracy_pct
    }

def promote_to_gold(staging_table, min_accuracy_pct=80.0):
    """
    Promotes validated classifications to gold table if accuracy meets threshold.
    Returns: True if promoted, False if validation failed
    """
    validation = validate_staging_table(staging_table)
    
    if validation['estimated_accuracy_pct'] >= min_accuracy_pct:
        print(f'\n✅ VALIDATION PASSED ({validation["estimated_accuracy_pct"]:.1f}% >= {min_accuracy_pct}%)')
        print(f'   Promoting {validation["total_classifications"]:,} classifications to gold...')
        
        spark.sql(f"""
            INSERT INTO workspace.superapp.gold_productos_categorias
            SELECT 
                id_producto,
                id_categoria,
                id_subcategoria,
                metodo,
                confianza,
                fecha_asignacion,
                usuario_asignacion,
                notas
            FROM {staging_table}
        """)
        
        print(f'   ✅ Promoted to gold_productos_categorias')
        return validation["total_classifications"]
    else:
        print(f'\n❌ VALIDATION FAILED ({validation["estimated_accuracy_pct"]:.1f}% < {min_accuracy_pct}%)')
        return 0

print('✅ Experiment isolation functions defined')

In [0]:
def find_resumable_run():
    """
    Finds the most recent staging table with unpromoted classifications.
    Returns: (staging_table, run_id, last_iteration) or (None, None, 0) if no resumable run
    """
    try:
        # Find all staging tables
        staging_tables = spark.sql("""
            SHOW TABLES IN workspace.superapp LIKE 'experiment_classifications_*'
        """).toPandas()
        
        if len(staging_tables) == 0:
            return None, None, 0
        
        # Check each staging table for unpromoted data (most recent first)
        for _, row in staging_tables.sort_values('tableName', ascending=False).iterrows():
            table_name = f"workspace.superapp.{row['tableName']}"
            
            # Get table info
            info = spark.sql(f"""
                SELECT 
                    COUNT(*) as total_classifications,
                    COUNT(DISTINCT id_producto) as unique_products,
                    COALESCE(MAX(iteration), 0) as max_iteration,
                    FIRST(run_id) as run_id,
                    MAX(fecha_asignacion) as last_updated
                FROM {table_name}
            """).collect()[0]
            
            if info['total_classifications'] == 0:
                continue  # Empty staging table, skip
            
            # Check if these products are already in gold (already promoted)
            already_promoted = spark.sql(f"""
                SELECT COUNT(*) as count
                FROM {table_name} stg
                INNER JOIN workspace.superapp.gold_productos_categorias gpc
                    ON stg.id_producto = gpc.id_producto
            """).collect()[0]['count']
            
            if already_promoted == info['total_classifications']:
                print(f'   ⏭️  {row["tableName"]}: Already promoted to gold (can be dropped)')
                continue
            
            # Found a staging table with unpromoted data!
            unpromoted = info['total_classifications'] - already_promoted
            print(f'   ✅ Found resumable run: {row["tableName"]}')
            print(f'      Run ID: {info["run_id"]}')
            print(f'      Total classifications: {info["total_classifications"]:,}')
            print(f'      Already promoted: {already_promoted:,}')
            print(f'      Unpromoted: {unpromoted:,}')
            print(f'      Last iteration: {info["max_iteration"]}')
            print(f'      Last updated: {info["last_updated"]}')
            
            return table_name, info['run_id'], info['max_iteration']
        
        print('   ℹ️  No resumable runs found (all staging tables are empty or already promoted)')
        return None, None, 0
        
    except Exception as e:
        print(f'   ⚠️  Error checking for resumable runs: {e}')
        return None, None, 0

print('✅ Función find_resumable_run() definida')

In [0]:
def load_centroids():
    """
    Carga centroids de MACRO y CATEGORIA desde las tablas gold.
    Retorna: (macro_matrix, macro_names, macro_id_map, categories_by_macro)
    """
    # Load MACRO centroids (9 macros)
    macro_pd = spark.table('workspace.superapp.gold_macro_centroids').toPandas()
    macro_pd['centroid'] = macro_pd['centroid_json'].apply(json.loads).apply(np.array)
    
    macro_matrix = np.array(macro_pd['centroid'].tolist())  # (9, 384)
    macro_names = macro_pd['nombre'].tolist()
    macro_id_map = dict(zip(macro_pd['nombre'], macro_pd['id_macro']))
    
    # Load CATEGORY centroids (139 categories) with macro mapping
    cat_pd = spark.sql("""
        SELECT 
            cc.id_categoria,
            cc.nombre,
            cc.centroid_json,
            c.id_macro
        FROM workspace.superapp.gold_category_centroids cc
        JOIN workspace.superapp.gold_categorias c ON cc.id_categoria = c.id_categoria
        WHERE c.id_macro IS NOT NULL
          AND cc.nombre NOT LIKE '\_OLD\_%'
          AND cc.nombre NOT LIKE '\_DELETED\_%'
    """).toPandas()
    
    cat_pd['centroid'] = cat_pd['centroid_json'].apply(json.loads).apply(np.array)
    
    # Group categories by macro
    categories_by_macro = {}
    for macro_id in macro_id_map.values():
        macro_cats = cat_pd[cat_pd['id_macro'] == macro_id]
        if len(macro_cats) > 0:
            categories_by_macro[macro_id] = {
                'names': macro_cats['nombre'].tolist(),
                'ids': macro_cats['id_categoria'].tolist(),
                'matrix': np.array(macro_cats['centroid'].tolist())
            }
    
    return macro_matrix, macro_names, macro_id_map, categories_by_macro

print('✅ Función load_centroids() definida')

In [0]:
from datetime import datetime
import numpy as np

def cosine_sim_batch(embeddings, centroid_matrix):
    """Calcula similitud coseno entre embeddings y centroids"""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normed = embeddings / (norms + 1e-10)
    return normed @ centroid_matrix.T

def classify_batch_phase1(batch_df, model, macro_matrix, macro_names, macro_id_map, categories_by_macro, all_categories_dict, 
                          macro_auto_threshold, category_auto_threshold, llm_macro_min, llm_macro_max, use_llm=False):
    """
    FASE 1: Calcula embeddings, similitudes, y prepara preguntas para LLM batch.
    NO llama al LLM todavía - solo acumula las preguntas.
    
    Args:
        batch_df: DataFrame con productos a clasificar
        model: modelo de embeddings
        macro_matrix, macro_names, macro_id_map: datos de macros
        categories_by_macro: categorías organizadas por macro
        all_categories_dict: ejemplos de categorías para LLM
        macro_auto_threshold: threshold para auto-asignar por macro (adaptive)
        category_auto_threshold: threshold para auto-asignar por categoría (adaptive)
        llm_macro_min: threshold mínimo para usar LLM
        llm_macro_max: threshold máximo para usar LLM
        use_llm: si usar LLM o no
    
    Retorna: (auto_assigned, llm_questions, product_context)
    - auto_assigned: productos que se auto-asignan sin LLM (alta confianza)
    - llm_questions: lista de preguntas para procesar en batch con Genie
    - product_context: dict con contexto de cada producto para uso posterior
    """
    auto_assigned = []
    llm_questions = []  # [{id_producto, tipo, prompt, context}, ...]
    product_context = {}  # {id_producto: {macro_id, cat_matrix, cat_names, cat_ids, ...}}
    
    # Generar embeddings
    embs = model.encode(batch_df['producto'].tolist(), batch_size=128, show_progress_bar=False)
    
    # LEVEL 1: Classify by MACRO
    macro_sims = cosine_sim_batch(embs, macro_matrix)
    macro_top_idx = macro_sims.argmax(axis=1)
    macro_top_scores = macro_sims.max(axis=1)
    
    for i, (_, row) in enumerate(batch_df.iterrows()):
        macro_score = float(macro_top_scores[i])
        macro_idx = macro_top_idx[i]
        macro_name_embedding = macro_names[macro_idx]
        id_producto = row['id_producto']
        producto = row['producto']
        
        # ========================================
        # PATH 1: FULL LLM (macro_sim < llm_macro_min)
        # ========================================
        if use_llm and macro_score < llm_macro_min:
            # Preparar pregunta de MACRO para LLM
            prompt = build_macro_prompt(producto, macro_names)
            
            llm_questions.append({
                'id_producto': id_producto,
                'producto': producto,
                'tipo': 'macro',
                'prompt': prompt,
                'embedding_macro': macro_name_embedding,
                'embedding_macro_sim': macro_score
            })
            
            # Guardar embedding del producto para fase 2
            product_context[id_producto] = {
                'embedding': embs[i],
                'embedding_macro': macro_name_embedding,
                'embedding_macro_sim': macro_score
            }
            
            continue
        
        # ========================================
        # PATH 2 & 3: Embedding decides MACRO
        # ========================================
        
        macro_id = macro_id_map[macro_name_embedding]
        
        # Check if macro has categories
        if macro_id not in categories_by_macro:
            # Skip - no categories in this macro
            continue
        
        macro_data = categories_by_macro[macro_id]
        cat_matrix = macro_data['matrix']
        cat_names = macro_data['names']
        cat_ids = macro_data['ids']
        
        cat_sims = cosine_sim_batch(embs[i:i+1], cat_matrix)[0]
        cat_top_idx = cat_sims.argmax()
        cat_top_score = float(cat_sims[cat_top_idx])
        cat_name = cat_names[cat_top_idx]
        cat_id = cat_ids[cat_top_idx]
        
        # ========================================
        # PATH 3: HIGH confidence (macro ≥ llm_macro_max, cat ≥ category_auto_threshold)
        # ========================================
        if macro_score >= llm_macro_max and cat_top_score >= category_auto_threshold:
            auto_assigned.append({
                'id_producto': id_producto,
                'id_categoria': int(cat_id),
                'id_subcategoria': None,
                'metodo': 'embedding_hierarchical_convergence',
                'confianza': cat_top_score,
                'fecha_asignacion': datetime.now(),
                'usuario_asignacion': 'system',
                'notas': f'macro={macro_name_embedding}({macro_score:.3f})|cat={cat_name}({cat_top_score:.3f})'
            })
            
        # ========================================
        # PATH 2: MEDIUM macro confidence (llm_macro_min ≤ macro < llm_macro_max) - LLM validates category
        # ========================================
        elif use_llm and llm_macro_min <= macro_score < llm_macro_max:
            # Get top N candidates with examples
            top_indices = cat_sims.argsort()[-3:][::-1]  # LLM_TOP_CANDIDATES = 3
            top_candidates = []
            for idx in top_indices:
                cat_candidate = cat_names[idx]
                sim = float(cat_sims[idx])
                examples = all_categories_dict.get(cat_candidate, [])
                top_candidates.append((cat_candidate, sim, examples))
            
            # Preparar pregunta de CATEGORIA para LLM
            prompt = build_category_prompt(producto, top_candidates, all_categories_dict)
            
            llm_questions.append({
                'id_producto': id_producto,
                'producto': producto,
                'tipo': 'categoria',
                'prompt': prompt,
                'embedding_macro': macro_name_embedding,
                'embedding_macro_sim': macro_score,
                'embedding_cat': cat_name,
                'embedding_cat_sim': cat_top_score
            })
            
            # Guardar contexto para fase 2
            product_context[id_producto] = {
                'macro_id': macro_id,
                'cat_names': cat_names,
                'cat_ids': cat_ids,
                'embedding_macro': macro_name_embedding,
                'embedding_macro_sim': macro_score,
                'embedding_cat': cat_name,
                'embedding_cat_sim': cat_top_score
            }
    
    return auto_assigned, llm_questions, product_context

print('✅ Función classify_batch_phase1() definida (acepta thresholds adaptativos)')

In [0]:
from datetime import datetime
import json
import pandas as pd

def process_llm_batch(llm_questions, product_context, macro_id_map, categories_by_macro, model):
    """
    FASE 2: Procesa todas las preguntas LLM en batch usando Genie (ai_query).
    
    Args:
        llm_questions: lista de preguntas preparadas en fase 1
        product_context: dict con contexto de cada producto
        macro_id_map: mapeo de nombre macro a ID
        categories_by_macro: dict con categorías por macro
        model: modelo de embeddings (para caso full_path)
    
    Returns:
        (auto_assigned, review_queue, stats)
    """
    if not llm_questions:
        return [], [], {'llm_called': 0, 'llm_success': 0, 'llm_full_path': 0, 'llm_category_only': 0}
    
    print(f'   🤖 Procesando {len(llm_questions)} preguntas LLM en batch con Genie...')
    
    # 1. Crear tabla temporal con preguntas
    questions_df = pd.DataFrame(llm_questions)
    spark_questions = spark.createDataFrame(questions_df)
    spark_questions.createOrReplaceTempView('llm_batch_questions')
    
    # 2. Ejecutar ai_query en batch
    results_df = spark.sql("""
        SELECT 
            id_producto,
            tipo,
            ai_query('superapp_endpoint', prompt) as respuesta,
            embedding_macro,
            embedding_macro_sim,
            embedding_cat,
            embedding_cat_sim
        FROM llm_batch_questions
    """).toPandas()
    
    print(f'   ✅ Genie procesó {len(results_df)} respuestas')
    
    # 3. Parse respuestas y clasificar
    auto_assigned = []
    review_queue = []
    stats = {'llm_called': len(llm_questions), 'llm_success': 0, 'llm_full_path': 0, 'llm_category_only': 0}
    
    for _, row in results_df.iterrows():
        id_producto = row['id_producto']
        tipo = row['tipo']
        respuesta_raw = row['respuesta']
        context = product_context.get(id_producto, {})
        
        # Parse JSON response
        try:
            respuesta = json.loads(respuesta_raw)
        except:
            # JSON parsing error - review queue
            review_queue.append({
                'id_producto': id_producto,
                'producto': llm_questions[0]['producto'],  # lookup from original
                'razon': 'llm_json_parse_error',
                'top_macro': row.get('embedding_macro'),
                'similitud_macro': row.get('embedding_macro_sim'),
                'top_categoria': row.get('embedding_cat'),
                'similitud': row.get('embedding_cat_sim'),
                'llm_sugerencia': respuesta_raw[:100],
                'estado': 'pendiente',
                'fecha_creacion': datetime.now()
            })
            continue
        
        # ========================================
        # TIPO: MACRO (full LLM path)
        # ========================================
        if tipo == 'macro':
            stats['llm_full_path'] += 1
            
            llm_macro = respuesta.get('macro', 'ninguna')
            llm_macro_conf = float(respuesta.get('confianza', 0.0))
            
            if llm_macro == 'ninguna' or llm_macro_conf < 0.7:
                # LLM uncertain - review queue
                review_queue.append({
                    'id_producto': id_producto,
                    'producto': next(q['producto'] for q in llm_questions if q['id_producto'] == id_producto),
                    'razon': 'llm_macro_low_confidence',
                    'top_macro': row.get('embedding_macro'),
                    'similitud_macro': row.get('embedding_macro_sim'),
                    'top_categoria': None,
                    'similitud': None,
                    'llm_sugerencia': f'{llm_macro}({llm_macro_conf:.2f})',
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
                continue
            
            # LLM confirmed macro - now decide category
            macro_id = macro_id_map.get(llm_macro)
            if not macro_id or macro_id not in categories_by_macro:
                review_queue.append({
                    'id_producto': id_producto,
                    'producto': next(q['producto'] for q in llm_questions if q['id_producto'] == id_producto),
                    'razon': 'llm_macro_not_found',
                    'top_macro': llm_macro,
                    'similitud_macro': llm_macro_conf,
                    'top_categoria': None,
                    'similitud': None,
                    'llm_sugerencia': llm_macro,
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
                continue
            
            # Use embedding to find best category within LLM-selected macro
            macro_data = categories_by_macro[macro_id]
            cat_matrix = macro_data['matrix']
            cat_names = macro_data['names']
            cat_ids = macro_data['ids']
            
            embedding = context['embedding']
            cat_sims = cosine_sim_batch(embedding.reshape(1, -1), cat_matrix)[0]
            cat_top_idx = cat_sims.argmax()
            cat_top_score = float(cat_sims[cat_top_idx])
            cat_name = cat_names[cat_top_idx]
            cat_id = cat_ids[cat_top_idx]
            
            if cat_top_score >= 0.60:  # Lower threshold for full LLM path
                auto_assigned.append({
                    'id_producto': id_producto,
                    'id_categoria': int(cat_id),
                    'id_subcategoria': None,
                    'metodo': 'embedding_hierarchical_llm_full_convergence',
                    'confianza': (llm_macro_conf + cat_top_score) / 2,  # Average
                    'fecha_asignacion': datetime.now(),
                    'usuario_asignacion': 'system',
                    'notas': f'llm_macro={llm_macro}({llm_macro_conf:.3f})|emb_cat={cat_name}({cat_top_score:.3f})'
                })
                stats['llm_success'] += 1
            else:
                review_queue.append({
                    'id_producto': id_producto,
                    'producto': next(q['producto'] for q in llm_questions if q['id_producto'] == id_producto),
                    'razon': 'llm_macro_ok_but_low_category',
                    'top_macro': llm_macro,
                    'similitud_macro': llm_macro_conf,
                    'top_categoria': cat_name,
                    'similitud': cat_top_score,
                    'llm_sugerencia': f'{llm_macro}>{cat_name}',
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
        
        # ========================================
        # TIPO: CATEGORIA (hybrid path)
        # ========================================
        elif tipo == 'categoria':
            stats['llm_category_only'] += 1
            
            llm_cat = respuesta.get('categoria', 'ninguna')
            llm_cat_conf = float(respuesta.get('confianza', 0.0))
            
            if llm_cat == 'ninguna' or llm_cat_conf < 0.7:
                # LLM uncertain - review queue
                review_queue.append({
                    'id_producto': id_producto,
                    'producto': next(q['producto'] for q in llm_questions if q['id_producto'] == id_producto),
                    'razon': 'llm_category_low_confidence',
                    'top_macro': context.get('embedding_macro'),
                    'similitud_macro': context.get('embedding_macro_sim'),
                    'top_categoria': context.get('embedding_cat'),
                    'similitud': context.get('embedding_cat_sim'),
                    'llm_sugerencia': f'{llm_cat}({llm_cat_conf:.2f})' if llm_cat != 'ninguna' else 'ninguna',
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
                continue
            
            # LLM confirmed category - find its ID
            cat_names = context['cat_names']
            cat_ids = context['cat_ids']
            
            llm_cat_id = None
            for idx, name in enumerate(cat_names):
                if name == llm_cat:
                    llm_cat_id = cat_ids[idx]
                    break
            
            if llm_cat_id:
                auto_assigned.append({
                    'id_producto': id_producto,
                    'id_categoria': int(llm_cat_id),
                    'id_subcategoria': None,
                    'metodo': 'embedding_hierarchical_llm_convergence',
                    'confianza': llm_cat_conf,
                    'fecha_asignacion': datetime.now(),
                    'usuario_asignacion': 'system',
                    'notas': f'emb_macro={context.get("embedding_macro")}({context.get("embedding_macro_sim"):.3f})|llm_cat={llm_cat}({llm_cat_conf:.3f})'
                })
                stats['llm_success'] += 1
            else:
                review_queue.append({
                    'id_producto': id_producto,
                    'producto': next(q['producto'] for q in llm_questions if q['id_producto'] == id_producto),
                    'razon': 'llm_category_not_found',
                    'top_macro': context.get('embedding_macro'),
                    'similitud_macro': context.get('embedding_macro_sim'),
                    'top_categoria': llm_cat,
                    'similitud': llm_cat_conf,
                    'llm_sugerencia': llm_cat,
                    'estado': 'pendiente',
                    'fecha_creacion': datetime.now()
                })
    
    return auto_assigned, review_queue, stats

print('✅ Función process_llm_batch() definida (procesa preguntas con Genie en batch)')

In [0]:
def rebuild_centroids():
    """
    Recalcula centroids inline usando todos los productos clasificados.
    Retorna True si exitoso, False si falla.
    """
    try:
        print('   🔧 Recalculando centroids...')
        
        # 1. Load all classified products with their categories
        labeled = spark.sql("""
            SELECT DISTINCT
                pc.id_producto,
                COALESCE(pc.id_subcategoria, pc.id_categoria) AS id_categoria_final,
                COALESCE(gcs.nombre, gc.nombre) AS categoria_nombre,
                sp.producto
            FROM workspace.superapp.gold_productos_categorias pc
            LEFT JOIN workspace.superapp.gold_categorias gc
                ON pc.id_categoria = gc.id_categoria
            LEFT JOIN workspace.superapp.gold_categorias gcs
                ON pc.id_subcategoria = gcs.id_categoria
            JOIN workspace.superapp.silver_prices sp
                ON pc.id_producto = sp.id_producto
            WHERE pc.id_categoria IS NOT NULL
        """).toPandas()
        
        if len(labeled) == 0:
            print('   ⚠️  No hay productos clasificados aún')
            return False
            
        print(f'   📊 Productos clasificados: {len(labeled):,}')
        
        # 2. Generate embeddings (using same model as classification)
        embeddings = model.encode(
            labeled['producto'].tolist(),
            show_progress_bar=False,
            batch_size=128
        )
        labeled['embedding'] = list(embeddings)
        
        # 3. Compute centroids per category
        centroids = []
        for cat_nombre, group in labeled.groupby('categoria_nombre'):
            vecs = np.array(group['embedding'].tolist())
            centroid = vecs.mean(axis=0)
            norm = np.linalg.norm(centroid)
            if norm > 1e-10:
                centroid = centroid / norm
            
            centroids.append({
                'id_categoria': int(group['id_categoria_final'].iloc[0]),
                'nombre': cat_nombre,
                'n_productos': len(group),
                'centroid_json': json.dumps(centroid.tolist())
            })
        
        centroids_pd = pd.DataFrame(centroids).sort_values('n_productos', ascending=False)
        print(f'   📌 Centroids calculados: {len(centroids_pd)}')
        
        # 4. Save to table
        (spark.createDataFrame(centroids_pd[['id_categoria', 'nombre', 'n_productos', 'centroid_json']])
              .write.mode('overwrite')
              .option('overwriteSchema', 'true')
              .saveAsTable('workspace.superapp.gold_category_centroids'))
        
        print('   ✅ Centroids guardados exitosamente')
        return True
        
    except Exception as e:
        print(f'   ❌ Error al recalcular centroids: {str(e)[:150]}')
        import traceback
        traceback.print_exc()
        return False

print('✅ Función rebuild_centroids() definida (inline)')

In [0]:
def validate_semantic_mismatches(staging_table=None):
    """
    Detecta errores semánticos obvios usando keywords en nombres de productos.
    Verifica que productos con keywords específicos estén en categorías apropiadas.
    
    Returns:
        dict: {
            'total_products': int,
            'mismatches': int,
            'rate': float,
            'by_category': dict  # {category_name: mismatch_count}
        }
    """
    # Define keyword patterns for common product categories
    # If keyword in product name, it should NOT be in these categories
    keyword_rules = [
        # (keyword_pattern, expected_category_pattern, forbidden_categories)
        ('cafe|coffee|cappuccino|espresso|latte', 'caf[eé]', ['Fernet', 'Jugos', 'Acelga', 'Felpudos', 'Cerezas']),
        ('yerba|mate', 'yerba|mate', ['Fernet', 'Jugos', 'Felpudos']),
        ('fideos|pasta|tallarin|spaghetti|ravioles', 'fideos|pasta', ['Fernet', 'Jugos', 'Cerezas']),
        ('bizcocho|galleta|cookie', 'galleta|bizcocho|dulce', ['Fernet', 'Jugos', 'Cerezas', 'Acelga']),
        ('aceite|oil', 'aceite|condimento', ['Fernet', 'Jugos', 'Felpudos']),
        ('manteca|mantequilla|ghee|butter', 'manteca|l[aá]cteo|dairy', ['Fernet', 'Jugos', 'Felpudos']),
        ('whiskas|purina|pedigree|maskota|perro|gato', 'mascota|pet|animal', ['Fernet', 'Acelga', 'Felpudos', 'Cerezas']),
        ('leche|milk|l[aá]cteo', 'leche|l[aá]cteo|dairy', ['Fernet', 'Jugos', 'Felpudos']),
        ('pan|bread', 'pan|panaderia|bakery', ['Fernet', 'Cerezas', 'Felpudos']),
        ('cargador|cable|celular|usb|samsung|iphone', 'electr[oó]nico|tecnolog[ií]a|cable|celular', ['Fernet', 'Acelga', 'Felpudos', 'Accesorios Tv'])
    ]
    
    # Build query - check gold or staging
    if staging_table:
        source_table = staging_table
        join_clause = f"{staging_table} gpc"
    else:
        source_table = "workspace.superapp.gold_productos_categorias"
        join_clause = "workspace.superapp.gold_productos_categorias gpc"
    
    # Build CASE statement for each rule
    case_statements = []
    for keyword, expected, forbidden in keyword_rules:
        forbidden_list = "', '".join(forbidden)
        case_statements.append(f"""
            WHEN LOWER(sp.producto) RLIKE '{keyword}' 
                 AND gc.nombre IN ('{forbidden_list}')
            THEN 1
        """)
    
    case_sql = "\n".join(case_statements)
    
    query = f"""
        SELECT 
            gc.nombre as categoria,
            COUNT(DISTINCT sp.id_producto) as total_products,
            SUM(CASE 
                {case_sql}
                ELSE 0
            END) as mismatch_count
        FROM {join_clause}
        JOIN workspace.superapp.gold_categorias gc 
            ON gpc.id_categoria = gc.id_categoria
        JOIN workspace.superapp.silver_prices sp 
            ON gpc.id_producto = sp.id_producto
        GROUP BY gc.nombre
        HAVING mismatch_count > 0
        ORDER BY mismatch_count DESC
    """
    
    mismatches_df = spark.sql(query).toPandas()
    
    # Get total products
    total_query = f"""
        SELECT COUNT(DISTINCT id_producto) as total
        FROM {source_table}
    """
    total_products = spark.sql(total_query).collect()[0]['total']
    
    # Calculate overall stats
    total_mismatches = mismatches_df['mismatch_count'].sum() if len(mismatches_df) > 0 else 0
    mismatch_rate = total_mismatches / total_products if total_products > 0 else 0
    
    by_category = {}
    if len(mismatches_df) > 0:
        for _, row in mismatches_df.iterrows():
            by_category[row['categoria']] = {
                'mismatch_count': int(row['mismatch_count']),
                'total_products': int(row['total_products']),
                'error_rate': row['mismatch_count'] / row['total_products']
            }
    
    return {
        'total_products': total_products,
        'mismatches': int(total_mismatches),
        'rate': float(mismatch_rate),
        'by_category': by_category
    }

print('✅ Función de validación semántica definida')

In [0]:
import numpy as np
from scipy.stats import entropy

def calculate_category_quality_metrics(source_table='workspace.superapp.gold_productos_categorias', top_n=10):
    """
    Calcula métricas de calidad sobre la distribución de categorías.
    
    Args:
        source_table: Tabla de clasificaciones (gold o staging)
        top_n: Número de categorías top a analizar
    
    Returns:
        dict: {
            'category_distribution': pd.DataFrame,  # Top N categories with stats
            'top_category_count': int,
            'top_category_pct': float,
            'category_entropy': float,
            'high_risk_categories': list,  # Categories with low std (too uniform)
            'total_classified': int
        }
    """
    # Get category distribution with confidence stats
    query = f"""
        SELECT 
            gc.nombre,
            COUNT(DISTINCT gpc.id_producto) as count,
            AVG(gpc.confianza) as avg_conf,
            STDDEV(gpc.confianza) as std_conf,
            MIN(gpc.confianza) as min_conf,
            MAX(gpc.confianza) as max_conf,
            PERCENTILE(gpc.confianza, 0.10) as p10_conf,
            PERCENTILE(gpc.confianza, 0.90) as p90_conf
        FROM {source_table} gpc
        JOIN workspace.superapp.gold_categorias gc 
            ON gpc.id_categoria = gc.id_categoria
        GROUP BY gc.nombre
        ORDER BY count DESC
        LIMIT {top_n}
    """
    
    category_dist = spark.sql(query).toPandas()
    
    # Get total for percentages
    total_query = f"""
        SELECT COUNT(DISTINCT id_producto) as total
        FROM {source_table}
    """
    total_classified = spark.sql(total_query).collect()[0]['total']
    
    if len(category_dist) == 0:
        return {
            'category_distribution': category_dist,
            'top_category_count': 0,
            'top_category_pct': 0,
            'category_entropy': 0,
            'high_risk_categories': [],
            'total_classified': 0
        }
    
    # Add percentage column
    category_dist['pct'] = category_dist['count'] / total_classified * 100
    
    # Calculate entropy (Shannon entropy of distribution)
    # Get full distribution for entropy calculation
    full_dist_query = f"""
        SELECT COUNT(DISTINCT id_producto) as count
        FROM {source_table}
        GROUP BY id_categoria
    """
    full_dist = spark.sql(full_dist_query).toPandas()
    probabilities = full_dist['count'] / full_dist['count'].sum()
    category_entropy = entropy(probabilities, base=2)
    
    # Identify high-risk categories (suspiciously uniform confidence)
    # Low std_conf suggests all products have similar confidence (potential centroid issue)
    high_risk = []
    for _, row in category_dist.iterrows():
        if pd.notna(row['std_conf']) and row['std_conf'] < 0.03:
            high_risk.append({
                'nombre': row['nombre'],
                'count': row['count'],
                'std_conf': row['std_conf'],
                'avg_conf': row['avg_conf']
            })
    
    return {
        'category_distribution': category_dist,
        'top_category_count': int(category_dist.iloc[0]['count']),
        'top_category_pct': float(category_dist.iloc[0]['pct']),
        'category_entropy': float(category_entropy),
        'high_risk_categories': high_risk,
        'total_classified': total_classified
    }

print('✅ Función de métricas de calidad definida')

In [0]:
print('🧪 TESTING QUALITY METRICS ON CURRENT GOLD DATA\n')
print('='*80)

# Test semantic validation
print('1️⃣ SEMANTIC VALIDATION (keyword-based error detection):')
semantic_results = validate_semantic_mismatches()  # Default: check gold table

print(f'   Total products: {semantic_results["total_products"]:,}')
print(f'   Mismatches: {semantic_results["mismatches"]:,} ({semantic_results["rate"]*100:.1f}%)')

if semantic_results['by_category']:
    print(f'\n   🚨 Top categories with semantic errors:')
    sorted_cats = sorted(
        semantic_results['by_category'].items(),
        key=lambda x: x[1]['mismatch_count'],
        reverse=True
    )[:5]
    
    for cat, data in sorted_cats:
        print(f"      {cat}: {data['mismatch_count']} errors ({data['error_rate']*100:.1f}% of {data['total_products']} products)")

# Test category quality metrics
print(f'\n\n2️⃣ CATEGORY QUALITY METRICS:')
quality_results = calculate_category_quality_metrics()

print(f'   Total classified: {quality_results["total_classified"]:,}')
print(f'   Top category: {quality_results["top_category_count"]:,} products ({quality_results["top_category_pct"]:.1f}%)')
print(f'   Distribution entropy: {quality_results["category_entropy"]:.2f}')

print(f'\n   📊 Top 5 categories:')
for _, row in quality_results['category_distribution'].head(5).iterrows():
    print(f"      {row['nombre']}: {row['count']:,} products ({row['pct']:.1f}%), "
          f"conf={row['avg_conf']:.1%} ±{row['std_conf']:.4f}")

if quality_results['high_risk_categories']:
    print(f'\n   ⚠️  HIGH RISK CATEGORIES (suspiciously uniform confidence):')
    for cat in quality_results['high_risk_categories'][:5]:
        print(f"      {cat['nombre']}: {cat['count']} products, "
              f"avg_conf={cat['avg_conf']:.1%}, std={cat['std_conf']:.4f}")

print('\n' + '='*80)
print('✅ Quality metrics working! These will now be logged to MLflow each iteration.')

In [0]:
# 🧪 LOOP WITH EXPERIMENT ISOLATION & STAGING TABLES
# Classifications go to staging first → validate → promote to gold if accurate

import uuid
from datetime import datetime
import mlflow
import mlflow.tracking._model_registry.utils

# Workaround for serverless: prevent MLflow from accessing spark.mlflow.modelRegistryUri
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks-uc"

# 🔧 SET UP MLFLOW EXPERIMENT
# Note: This notebook is in a Git repo, so we use the default experiment
# (custom experiments cannot be created in Git folders)
print('🔧 Setting up MLflow tracking...')
experiment_id = None
try:
    # Get the default experiment for this notebook (auto-created by Databricks)
    experiment = mlflow.get_experiment_by_name(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
    if experiment:
        experiment_id = experiment.experiment_id
        print(f'✅ Using default experiment for this notebook (ID: {experiment_id})')
    else:
        print('⚠️  Default experiment not found, will be created on first run')
except Exception as exp_err:
    print(f'⚠️  Experiment lookup failed: {exp_err}')

def _mlf_param(key, value):
    try: mlflow.log_param(key, value)
    except Exception: pass

def _mlf_metric(key, value, step=None):
    try:
        mlflow.log_metric(key, value, step=step) if step is not None else mlflow.log_metric(key, value)
    except Exception: pass

# 🔄 CHECK FOR RESUMABLE RUN
print('\n🔍 Checking for interrupted runs...')
resumable_staging, resumable_run_id, last_iteration = find_resumable_run()

if resumable_staging:
    print(f'\n🔄 RESUMING FROM INTERRUPTED RUN')
    staging_table = resumable_staging
    # Keep the old run_id for staging table reference, but we'll create a NEW MLflow run
    old_run_id = resumable_run_id
    start_iteration = last_iteration  # Will continue from last_iteration + 1
    is_resume = True
    print(f'   Staging table: {staging_table}')
    print(f'   Old Run ID: {old_run_id}')
    print(f'   Resuming from iteration: {start_iteration + 1}')
else:
    print(f'\n✨ STARTING NEW RUN')
    is_resume = False
    start_iteration = 0
    old_run_id = None

print('\n🔧 Iniciando MLflow tracking...')
_mlflow_active = False
try:
    # ALWAYS create a NEW MLflow run (even when resuming staging table work)
    # For Git repos, explicitly pass experiment_id to avoid permission issues
    run_name = f"classification_convergence_resumed_{old_run_id}" if is_resume else f"classification_convergence_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    if experiment_id:
        _mlf_run = mlflow.start_run(run_name=run_name, experiment_id=experiment_id)
    else:
        _mlf_run = mlflow.start_run(run_name=run_name)
    
    run_id = _mlf_run.info.run_id
    
    if is_resume:
        print(f'✅ MLflow run created (new tracking for resumed work): {run_id}')
        print(f'   Note: Resuming staging table from old run {old_run_id}')
    else:
        print(f'✅ MLflow Run ID: {run_id}')
    
    _mlflow_active = True
    
    # Get experiment ID for UI link
    exp_id = _mlf_run.info.experiment_id
    print(f'   📊 View in MLflow UI: /ml/experiments/{exp_id}/runs/{run_id}')
except Exception as _e:
    if not is_resume:
        run_id = uuid.uuid4().hex[:8]
    else:
        run_id = old_run_id
    print(f'⚠️  MLflow unavailable ({_e.__class__.__name__}: {str(_e)[:100]}) — continuing without tracking. run_id={run_id}')

try:
    # 🧪 CREATE OR REUSE EXPERIMENT STAGING TABLE
    print(f'\n🧪 EXPERIMENT ISOLATION ENABLED')
    
    if not is_resume:
        staging_table = create_experiment_staging_table(run_id)
    else:
        print(f'   Reusing staging table: {staging_table}')
    
    _mlf_param("staging_table", staging_table)
    _mlf_param("is_resume", is_resume)
    if is_resume:
        _mlf_param("resumed_from_run", old_run_id)
        _mlf_param("resumed_from_iteration", start_iteration)
    
    # Dynamic thresholds - will adjust based on performance
    current_macro_auto = MACRO_AUTO
    current_category_auto = CATEGORY_AUTO
    current_llm_min = LLM_MACRO_MIN
    current_llm_max = LLM_MACRO_MAX
    
    print(f'\n🎯 THRESHOLDS INICIALES (adaptativos):')
    print(f'   MACRO_AUTO:     {current_macro_auto:.2f} (se ajustará según performance)')
    print(f'   CATEGORY_AUTO:  {current_category_auto:.2f} (se ajustará según performance)')
    print(f'   LLM_MIN:        {current_llm_min:.2f}')
    print(f'   LLM_MAX:        {current_llm_max:.2f}')
    
    # Log parameters
    _mlf_param("macro_auto_initial", current_macro_auto)
    _mlf_param("category_auto_initial", current_category_auto)
    _mlf_param("batch_size", BATCH_SIZE)
    _mlf_param("max_iterations", MAX_ITERATIONS)
    _mlf_param("rebuild_centroids_every", REBUILD_CENTROIDS_EVERY)
    _mlf_param("use_llm", USE_LLM)
    _mlf_param("llm_model", LLM_MODEL)
    _mlf_param("embedding_model", EMBEDDING_MODEL)
    
    # Load embedding model
    print('\n🧠 Cargando modelo e5-small...')
    model = SentenceTransformer(EMBEDDING_MODEL)
    print('✅ Modelo e5-small cargado')
    
    # Load initial centroids
    print('\n🔧 Cargando centroids iniciales...')
    macro_matrix, macro_names, macro_id_map, categories_by_macro = load_centroids()
    print(f'✅ Centroids cargados: {len(macro_names)} macros, {sum(len(v["names"]) for v in categories_by_macro.values())} categorías')
    
    # Load all categories with examples (for LLM)
    print('\n🔧 Cargando ejemplos de categorías para LLM...')
    cat_examples = spark.sql("""
        SELECT 
            c.nombre,
            get(COLLECT_LIST(sp.producto), 0) as producto
        FROM workspace.superapp.gold_categorias c
        LEFT JOIN workspace.superapp.gold_productos_categorias gpc ON c.id_categoria = gpc.id_categoria
        LEFT JOIN workspace.superapp.silver_prices sp ON gpc.id_producto = sp.id_producto
        WHERE c.nivel = 'categoria' AND c.is_active = TRUE
        GROUP BY c.nombre
    """)
    
    all_categories_dict = {}
    for _, row in cat_examples.toPandas().iterrows():
        if pd.notna(row['producto']):
            all_categories_dict[row['nombre']] = row['producto']
    
    print(f'✅ Cargados ejemplos de {len(all_categories_dict)} categorías')
    
    # 📊 Get initial baseline counts for MLflow tracking
    initial_classified = spark.sql("""
        SELECT COUNT(DISTINCT id_producto) as count
        FROM workspace.superapp.gold_productos_categorias
    """).collect()[0]['count']
    
    initial_unclassified = spark.sql("""
        SELECT COUNT(DISTINCT sp.id_producto) as count
        FROM workspace.superapp.silver_prices sp
        LEFT JOIN workspace.superapp.gold_productos_categorias gpc 
            ON sp.id_producto = gpc.id_producto
        WHERE gpc.id_producto IS NULL
    """).collect()[0]['count']
    
    total_products = initial_classified + initial_unclassified
    
    print(f'\n📊 BASELINE INICIAL:')
    print(f'   Total productos únicos: {total_products:,}')
    print(f'   Ya clasificados:        {initial_classified:,} ({initial_classified/total_products*100:.1f}%)')
    print(f'   Por clasificar:         {initial_unclassified:,} ({initial_unclassified/total_products*100:.1f}%)')
    
    # Count products assigned to the catch-all unclassified category
    initial_unclassified_cat = spark.sql(f"""
        SELECT COUNT(DISTINCT gpc.id_producto) as count
        FROM workspace.superapp.gold_productos_categorias gpc
        JOIN workspace.superapp.gold_categorias gc ON gpc.id_categoria = gc.id_categoria
        WHERE LOWER(gc.nombre) = LOWER('{UNCLASSIFIED_CATEGORY_NAME}')
    """).collect()[0]['count']
    
    # Log baseline to MLflow (only for new runs, not resumes)
    if not is_resume:
        _mlf_param("total_products", total_products)
        _mlf_param("initial_classified", initial_classified)
        _mlf_param("initial_unclassified", initial_unclassified)
        _mlf_metric("total_products", total_products)
        _mlf_metric("classified_products", initial_classified, step=0)
        _mlf_metric("unclassified_products", initial_unclassified, step=0)
        _mlf_metric("classification_pct", initial_classified/total_products*100 if total_products > 0 else 0, step=0)
        _mlf_metric("unclassified_category_count", initial_unclassified_cat, step=0)
    
    # Tracking variables
    iteration_results = []
    cumulative_classified = 0
    previous_top_5_counts = {}  # Track category growth
    prev_unclassified = initial_unclassified       # For convergence delta
    prev_unclassified_cat = initial_unclassified_cat  # For escape rate
    
    print('\n' + '='*80)
    if is_resume:
        print(f'🔄 RESUMIENDO LOOP DE CONVERGENCIA desde iteración {start_iteration + 1}')
    else:
        print('🚀 INICIANDO LOOP DE CONVERGENCIA (CON STAGING TABLES)')
    print('='*80)
    
    for i in range(start_iteration, MAX_ITERATIONS):
        print(f'\n--- Iteración {i+1}/{MAX_ITERATIONS} ---')
        print(f'   Thresholds: MACRO={current_macro_auto:.2f}, CATEGORY={current_category_auto:.2f}, LLM_MIN={current_llm_min:.2f}, LLM_MAX={current_llm_max:.2f}')
        
        # Get unclassified products - EXCLUDE both gold AND staging
        # ✅ FIX: Use GROUP BY to get exactly ONE row per unique id_producto
        # This handles duplicates in silver_prices (same product across stores/dates/descriptions)
        unclassified = spark.sql(f"""
            SELECT sp.id_producto, FIRST(sp.producto) as producto
            FROM workspace.superapp.silver_prices sp
            LEFT JOIN workspace.superapp.gold_productos_categorias gpc 
                ON sp.id_producto = gpc.id_producto
            LEFT JOIN {staging_table} stg
                ON sp.id_producto = stg.id_producto
            WHERE gpc.id_producto IS NULL
              AND stg.id_producto IS NULL
            GROUP BY sp.id_producto
        """)
        
        unclassified_count = unclassified.count()
        
        # Also show total classified count for context
        classified_count = spark.sql("""
            SELECT COUNT(DISTINCT id_producto) as count
            FROM workspace.superapp.gold_productos_categorias
        """).collect()[0]['count']
        
        # Count products in staging (not yet validated/promoted)
        staging_count = spark.sql(f"""
            SELECT COUNT(DISTINCT id_producto) as count
            FROM {staging_table}
        """).collect()[0]['count']
        
        print(f'📊 Productos clasificados (gold): {classified_count:,}')
        print(f'   Productos en staging (pendientes): {staging_count:,}')
        print(f'   Productos no clasificados: {unclassified_count:,}')
        
        # 📊 Log progress to MLflow
        _mlf_metric("classified_products", classified_count, step=i+1)
        _mlf_metric("staging_products", staging_count, step=i+1)
        _mlf_metric("unclassified_products", unclassified_count, step=i+1)
        _mlf_metric("classification_pct", classified_count/total_products*100 if total_products > 0 else 0, step=i+1)
        _mlf_metric("threshold_category_auto", current_category_auto, step=i+1)
        _mlf_metric("unclassified_delta", prev_unclassified - unclassified_count, step=i+1)
        prev_unclassified = unclassified_count
        
        if unclassified_count == 0:
            print('✅ No hay más productos para clasificar')
            break
        
        # Sample batch
        if unclassified_count < BATCH_SIZE:
            print(f'⚠️  Solo {unclassified_count} productos restantes')
            sample = unclassified.toPandas()
        else:
            sample = unclassified.sample(fraction=min(1.0, BATCH_SIZE/unclassified_count)).limit(BATCH_SIZE).toPandas()
        
        print(f'Batch size: {len(sample)}')
        
        # PHASE 1: Classify with embeddings (PASS ADAPTIVE THRESHOLDS)
        print(f'\n🔍 FASE 1: Clasificando con embeddings...')
        auto_assigned_phase1, llm_questions, product_context = classify_batch_phase1(
            sample, model, macro_matrix, macro_names, macro_id_map, categories_by_macro, all_categories_dict,
            macro_auto_threshold=current_llm_max,  # Use current_llm_max as high confidence threshold
            category_auto_threshold=current_category_auto,  # Use adaptive category threshold
            llm_macro_min=current_llm_min,
            llm_macro_max=current_llm_max,
            use_llm=USE_LLM
        )
        
        print(f'   ✅ Auto-asignados (embeddings): {len(auto_assigned_phase1)}')
        print(f'   🤖 Preguntas LLM generadas: {len(llm_questions)}')
        
        # PHASE 2: Process LLM batch
        auto_assigned_llm = []
        review_queue = []
        llm_stats = {'llm_called': 0, 'llm_success': 0, 'llm_full_path': 0, 'llm_category_only': 0}
        
        if USE_LLM and llm_questions:
            print(f'\n🤖 FASE 2: Ejecutando {len(llm_questions)} preguntas LLM INLINE...')
            auto_assigned_llm, review_queue, llm_stats = process_llm_batch(
                llm_questions, product_context, macro_id_map, categories_by_macro, model
            )
        
        # Combine auto-assigned
        auto_assigned = auto_assigned_phase1 + auto_assigned_llm
        cumulative_classified += len(auto_assigned)
        
        # Confidence distribution of products classified this iteration
        if auto_assigned:
            confs = sorted([p.get('confianza', 0.0) for p in auto_assigned])
            n = len(confs)
            _mlf_metric("iter_avg_confidence", sum(confs) / n, step=i+1)
            _mlf_metric("iter_p10_confidence", confs[max(0, int(n * 0.10))], step=i+1)
            _mlf_metric("iter_p90_confidence", confs[min(n - 1, int(n * 0.90))], step=i+1)
        
        # Results
        print(f'\n📊 RESULTADOS ITERACIÓN {i+1}:')
        print(f'   ✅ Total auto-asignados: {len(auto_assigned)} ({len(auto_assigned)/len(sample)*100:.1f}%)')
        print(f'      - Por embeddings: {len(auto_assigned_phase1)}')
        print(f'      - Por LLM: {len(auto_assigned_llm)}')
        print(f'   ⚠️  Review queue: {len(review_queue)} ({len(review_queue)/len(sample)*100:.1f}%)')
        print(f'   📈 Progreso acumulado en esta sesión: {cumulative_classified} productos')
        
        # Save to STAGING (use the staging_table run_id for consistency)
        saved_count = save_to_staging(auto_assigned, staging_table, old_run_id if is_resume else run_id, iteration=i+1)
        
        # 📊 Log iteration metrics to MLflow
        _mlf_metric("batch_auto_assigned", len(auto_assigned), step=i+1)
        _mlf_metric("batch_auto_assigned_embeddings", len(auto_assigned_phase1), step=i+1)
        _mlf_metric("batch_auto_assigned_llm", len(auto_assigned_llm), step=i+1)
        _mlf_metric("batch_review_queue", len(review_queue), step=i+1)
        _mlf_metric("batch_auto_rate_pct", len(auto_assigned)/len(sample)*100 if len(sample) > 0 else 0, step=i+1)
        _mlf_metric("cumulative_classified_this_run", cumulative_classified, step=i+1)
        
        # LLM path quality metrics
        _mlf_metric("llm_called", llm_stats['llm_called'], step=i+1)
        if llm_stats['llm_called'] > 0:
            _mlf_metric("llm_success_rate_pct", llm_stats['llm_success'] / llm_stats['llm_called'] * 100, step=i+1)
            _mlf_metric("llm_full_path_rate_pct", llm_stats['llm_full_path'] / llm_stats['llm_called'] * 100, step=i+1)
        
        # Review queue breakdown by failure reason
        rq_reasons = {}
        for _p in review_queue:
            _r = _p.get('razon', 'unknown')
            rq_reasons[_r] = rq_reasons.get(_r, 0) + 1
        for _reason, _cnt in rq_reasons.items():
            _mlf_metric(f'review_{_reason}', _cnt, step=i+1)
        
        # ========================================
        # 🎯 QUALITY METRICS - Track category distribution and detect issues
        # ========================================
        print(f'\n🎯 QUALITY METRICS:')
        
        try:
            # 1. Category Distribution Metrics
            quality_metrics = calculate_category_quality_metrics(staging_table, top_n=10)
            cat_dist = quality_metrics['category_distribution']
            
            if len(cat_dist) > 0:
                # Log top 5 categories
                top_5_str = ", ".join([f"{row['nombre']}({row['count']})" 
                                       for _, row in cat_dist.head(5).iterrows()])
                _mlf_param(f"top_5_categories_iter_{i+1}", top_5_str[:250])  # Truncate if too long
                
                # Log distribution metrics
                _mlf_metric("top_category_count", quality_metrics['top_category_count'], step=i+1)
                _mlf_metric("top_category_pct", quality_metrics['top_category_pct'], step=i+1)
                _mlf_metric("category_entropy", quality_metrics['category_entropy'], step=i+1)
                _mlf_metric("high_risk_category_count", len(quality_metrics['high_risk_categories']), step=i+1)
                
                print(f'   📊 Top 5 categorías: {top_5_str}')
                print(f'   📈 Top category: {quality_metrics["top_category_pct"]:.1f}% of products')
                print(f'   🎲 Entropy: {quality_metrics["category_entropy"]:.2f}')
                
                # Log confidence stats for top 5 categories
                for idx, row in cat_dist.head(5).iterrows():
                    cat_name = row['nombre'].replace(' ', '_').replace('/', '_')[:50]
                    _mlf_metric(f"cat_{cat_name}_count", row['count'], step=i+1)
                    _mlf_metric(f"cat_{cat_name}_conf_mean", row['avg_conf'], step=i+1)
                    if pd.notna(row['std_conf']):
                        _mlf_metric(f"cat_{cat_name}_conf_std", row['std_conf'], step=i+1)
                
                # Track category growth
                current_top_5_counts = {row['nombre']: row['count'] for _, row in cat_dist.head(5).iterrows()}
                if previous_top_5_counts:
                    for cat_name, current_count in current_top_5_counts.items():
                        if cat_name in previous_top_5_counts:
                            growth = (current_count - previous_top_5_counts[cat_name]) / previous_top_5_counts[cat_name]
                            cat_name_safe = cat_name.replace(' ', '_').replace('/', '_')[:50]
                            _mlf_metric(f"cat_{cat_name_safe}_growth_pct", growth * 100, step=i+1)
                previous_top_5_counts = current_top_5_counts
                
                # Alerts for high-risk categories
                if quality_metrics['high_risk_categories']:
                    print(f'\n   ⚠️  HIGH RISK CATEGORIES (uniform confidence, possible centroid issues):')
                    for cat in quality_metrics['high_risk_categories']:
                        print(f'      - {cat["nombre"]}: {cat["count"]} products, '
                              f'avg_conf={cat["avg_conf"]:.1%}, std={cat["std_conf"]:.4f}')
                        cat_name_safe = cat['nombre'].replace(' ', '_').replace('/', '_')[:50]
                        _mlf_metric(f"cat_{cat_name_safe}_variance_alert", 1, step=i+1)
                
                # Alert if top category is too dominant
                if quality_metrics['top_category_pct'] > 15.0:
                    print(f'   ⚠️  ALERT: Top category has {quality_metrics["top_category_pct"]:.1f}% of products (>15%)')
                    _mlf_metric("alert_dominant_category", 1, step=i+1)
            
            # 2. Semantic Validation Metrics
            semantic_val = validate_semantic_mismatches(staging_table)
            
            _mlf_metric("semantic_mismatch_count", semantic_val['mismatches'], step=i+1)
            _mlf_metric("semantic_mismatch_rate", semantic_val['rate'] * 100, step=i+1)
            
            if semantic_val['rate'] > 0:
                print(f'   🚨 Semantic mismatches: {semantic_val["mismatches"]} ({semantic_val["rate"]*100:.1f}%)')
                
                # Log top 3 categories with most mismatches
                if semantic_val['by_category']:
                    top_3_mismatches = sorted(
                        semantic_val['by_category'].items(),
                        key=lambda x: x[1]['mismatch_count'],
                        reverse=True
                    )[:3]
                    
                    mismatch_str = ", ".join([f"{cat}({data['mismatch_count']})" 
                                              for cat, data in top_3_mismatches])
                    _mlf_param(f"top_3_mismatch_categories_iter_{i+1}", mismatch_str[:250])
                    print(f'      Top mismatched categories: {mismatch_str}')
                    
                    # Log per-category error rates
                    for cat, data in top_3_mismatches:
                        cat_name_safe = cat.replace(' ', '_').replace('/', '_')[:50]
                        _mlf_metric(f"cat_{cat_name_safe}_error_rate", data['error_rate'] * 100, step=i+1)
                
                # Alert if semantic mismatch rate is high
                if semantic_val['rate'] > 0.10:
                    print(f'   ⚠️  ALERT: High semantic mismatch rate ({semantic_val["rate"]*100:.1f}% > 10%)')
                    _mlf_metric("alert_high_semantic_errors", 1, step=i+1)
            else:
                print(f'   ✅ No semantic mismatches detected')
        
            # 3. Unclassified Category Metrics (catch-all vs truly classified)
            try:
                unclassified_cat_count = spark.sql(f"""
                    SELECT COUNT(DISTINCT gpc.id_producto) as count
                    FROM workspace.superapp.gold_productos_categorias gpc
                    JOIN workspace.superapp.gold_categorias gc ON gpc.id_categoria = gc.id_categoria
                    WHERE LOWER(gc.nombre) = LOWER('{UNCLASSIFIED_CATEGORY_NAME}')
                """).collect()[0]['count']
                
                escaped = max(0, prev_unclassified_cat - unclassified_cat_count)
                escape_rate = escaped / prev_unclassified_cat * 100 if prev_unclassified_cat > 0 else 0.0
                truly_classified = classified_count - unclassified_cat_count
                
                _mlf_metric("unclassified_category_count", unclassified_cat_count, step=i+1)
                _mlf_metric("unclassified_escape_rate_pct", escape_rate, step=i+1)
                _mlf_metric("truly_classified_pct", truly_classified / total_products * 100 if total_products > 0 else 0.0, step=i+1)
                
                prev_unclassified_cat = unclassified_cat_count
                
                if unclassified_cat_count > 0:
                    print(f'   🗂️  Categoría "{UNCLASSIFIED_CATEGORY_NAME}": {unclassified_cat_count:,} productos')
                    print(f'   📤 Escape rate: {escape_rate:.1f}%')
                    print(f'   ✅ Verdaderamente clasificados: {truly_classified:,} ({truly_classified/total_products*100:.1f}%)')
            except Exception as uncat_err:
                print(f'   ⚠️  Unclassified category metrics error: {uncat_err}')
        
        except Exception as quality_err:
            print(f'   ⚠️  Quality metrics error: {quality_err}')
            _mlf_param(f"quality_metrics_error_iter_{i+1}", str(quality_err)[:200])
        
        # Track results
        iteration_results.append({
            'iteration': i+1,
            'batch_size': len(sample),
            'auto_assigned': len(auto_assigned),
            'auto_phase1': len(auto_assigned_phase1),
            'auto_phase2': len(auto_assigned_llm),
            'review_queue': len(review_queue),
            'macro_auto': current_macro_auto,
            'category_auto': current_category_auto,
            'llm_min': current_llm_min
        })
        
        # Save review queue
        if review_queue:
            review_pdf = pd.DataFrame(review_queue)
            review_df = spark.createDataFrame(review_pdf)
            review_df.write.mode('append').saveAsTable('workspace.superapp.gold_review_queue')
            print(f'   💾 Review queue guardada: {len(review_queue)} productos')
        
        # ADAPTIVE THRESHOLDS
        auto_rate = len(auto_assigned) / len(sample) if len(sample) > 0 else 0
        
        if auto_rate < 0.40:
            current_category_auto = max(0.50, current_category_auto - 0.05)
            print(f'   📉 Reduciendo CATEGORY_AUTO a {current_category_auto:.2f} (auto rate: {auto_rate:.1%})')
        elif auto_rate > 0.60:
            current_category_auto = min(0.80, current_category_auto + 0.05)
            print(f'   📈 Aumentando CATEGORY_AUTO a {current_category_auto:.2f} (auto rate: {auto_rate:.1%})')
        
        # Recalculate centroids every N iterations
        if len(auto_assigned) > 0 and (i + 1) % REBUILD_CENTROIDS_EVERY == 0:
            print(f'\n🔄 Recalculando centroids (cada {REBUILD_CENTROIDS_EVERY} iteraciones)...')
            macro_matrix, macro_names, macro_id_map, categories_by_macro = load_centroids()
            print(f'✅ Centroids actualizados')
    
    # 📊 FINAL SUMMARY
    print('\n' + '='*80)
    print('✅ CONVERGENCE LOOP COMPLETADO')
    print('='*80)
    
    final_classified = spark.sql("""
        SELECT COUNT(DISTINCT id_producto) as count
        FROM workspace.superapp.gold_productos_categorias
    """).collect()[0]['count']
    
    final_staging = spark.sql(f"""
        SELECT COUNT(DISTINCT id_producto) as count
        FROM {staging_table}
    """).collect()[0]['count']
    
    final_unclassified = spark.sql("""
        SELECT COUNT(DISTINCT sp.id_producto) as count
        FROM workspace.superapp.silver_prices sp
        LEFT JOIN workspace.superapp.gold_productos_categorias gpc 
            ON sp.id_producto = gpc.id_producto
        WHERE gpc.id_producto IS NULL
    """).collect()[0]['count']
    
    print(f'\n📊 RESULTADOS FINALES:')
    print(f'   Total productos: {total_products:,}')
    print(f'   Clasificados (gold): {final_classified:,} ({final_classified/total_products*100:.1f}%)')
    print(f'   En staging (pendiente validación): {final_staging:,}')
    print(f'   No clasificados: {final_unclassified:,} ({final_unclassified/total_products*100:.1f}%)')
    print(f'\n   Productos clasificados en esta sesión: {cumulative_classified:,}')
    print(f'   Iteraciones completadas: {len(iteration_results)}')
    
    # Log final metrics
    _mlf_metric("final_classified_products", final_classified)
    _mlf_metric("final_staging_products", final_staging)
    _mlf_metric("final_unclassified_products", final_unclassified)
    _mlf_metric("final_classification_pct", final_classified/total_products*100 if total_products > 0 else 0)
    _mlf_metric("session_classified_count", cumulative_classified)
    _mlf_metric("iterations_completed", len(iteration_results))
    
    # Summary table
    if iteration_results:
        results_df = pd.DataFrame(iteration_results)
        print('\n📊 RESUMEN POR ITERACIÓN:')
        print(results_df.to_string(index=False))
        
        # Save to table for analysis
        spark_results = spark.createDataFrame(results_df)
        spark_results.write.mode('append').saveAsTable('workspace.superapp.gold_convergence_metrics')
        print('\n💾 Métricas guardadas en gold_convergence_metrics')
    
    print(f'\n🎯 PRÓXIMO PASO: Validar productos en staging table y promover a gold')
    print(f'   Staging table: {staging_table}')
    if is_resume:
        print(f'   Original Run ID: {old_run_id}')
    print(f'   MLflow Run ID: {run_id}')
    
except KeyboardInterrupt:
    print('\n⚠️  LOOP INTERRUMPIDO POR USUARIO')
    print(f'   Staging table: {staging_table}')
    if is_resume:
        print(f'   Original Run ID: {old_run_id}')
    print(f'   MLflow Run ID: {run_id}')
    print(f'   Última iteración completada: {i+1}')
    print(f'   Productos clasificados en esta sesión: {cumulative_classified}')
    print('\n   ℹ️  Puedes resumir ejecutando este cell de nuevo')
    raise
except Exception as e:
    print(f'\n❌ ERROR: {e}')
    import traceback
    traceback.print_exc()
    print(f'\n   Staging table: {staging_table}')
    if is_resume:
        print(f'   Original Run ID: {old_run_id}')
    print(f'   MLflow Run ID: {run_id}')
    raise
finally:
    if _mlflow_active:
        try:
            mlflow.end_run()
            print('\n✅ MLflow run ended')
        except:
            pass

In [0]:
%sql
-- Quick test to see if AI functions are available
SELECT ai_gen('Say hello') as test

In [0]:
# Start with EXACT working pattern from ML notebook
import mlflow

print('🧪 Testing minimal MLflow pattern...')

with mlflow.start_run():
    # Just log basic params/metrics like working notebook
    mlflow.log_param("test_param", "value1")
    mlflow.log_metric("test_metric", 0.99)
    
    print('✅ MLflow basic logging works!')
    print(f'   Run ID: {mlflow.active_run().info.run_id}')

In [0]:
# Create DataFrame with results
import pandas as pd
import matplotlib.pyplot as plt

results_df = pd.DataFrame(iteration_results)

print('
📊 RESULTADOS DETALLADOS POR ITERACION')
print('='*100)
print(results_df[[
    'iteration', 'batch_size', 'auto_assigned', 'auto_phase1', 'auto_phase2',
    'review_queue', 'macro_auto', 'category_auto'
]].to_string(index=False))

print('
🔍 ANALISIS')
print('='*100)

if results_df['auto_assigned'].sum() > 0:
    total_auto = results_df['auto_assigned'].sum()
    total_phase1 = results_df['auto_phase1'].sum()
    total_phase2 = results_df['auto_phase2'].sum()
    total_review = results_df['review_queue'].sum()
    total_processed = results_df['batch_size'].sum()

    print(f'✅ Total auto-asignados: {total_auto}')
    print(f'   - Por embeddings (alta confianza): {total_phase1} ({total_phase1/total_auto*100:.1f}%)')
    print(f'   - Por LLM batch:                  {total_phase2} ({total_phase2/total_auto*100:.1f}%)')
    print(f'
📋 Review queue: {total_review:,} ({total_review/total_processed*100:.1f}% del total)')

    if len(results_df) > 1:
        print(f'
🎯 Evolucion de thresholds:')
        print(f'   MACRO_AUTO:    {results_df["macro_auto"].iloc[0]:.2f} -> {results_df["macro_auto"].iloc[-1]:.2f}')
        print(f'   CATEGORY_AUTO: {results_df["category_auto"].iloc[0]:.2f} -> {results_df["category_auto"].iloc[-1]:.2f}')
else:
    print('❌ NO hubo productos auto-asignados')
    print('   -> Los thresholds son muy altos o los embeddings no capturan bien estos productos')

print('='*100)


In [0]:
# Mostrar ejemplos de preguntas LLM y respuestas
if 'llm_batch_questions' in dir():
    print('\n🔍 EJEMPLOS DE PREGUNTAS LLM (Batch Processing)')
    print('='*100)
    
    # Leer las preguntas que quedaron en la tabla temporal
    try:
        questions_sample = spark.sql("""
            SELECT 
                id_producto,
                producto,
                tipo,
                LEFT(prompt, 200) as prompt_preview,
                embedding_macro,
                embedding_macro_sim
            FROM llm_batch_questions
            LIMIT 5
        """).toPandas()
        
        print(f'\nMostrando {len(questions_sample)} ejemplos de preguntas:')
        print('-'*100)
        
        for i, row in questions_sample.iterrows():
            print(f"\n{i+1}. Producto: {row['producto']}")
            print(f"   Tipo: {row['tipo']}")
            print(f"   Embedding macro: {row['embedding_macro']} (sim={row['embedding_macro_sim']:.3f})")
            print(f"   Prompt preview: {row['prompt_preview']}...")
            print('-'*100)
    
    except Exception as e:
        print(f'No se pudieron leer ejemplos: {e}')

print('\n✅ Análisis completo - revisar resultados arriba')

In [0]:
%sql
SELECT
    COUNT(DISTINCT sp.id_producto)  AS total_products,
    COUNT(DISTINCT gpc.id_producto) AS classified,
    COUNT(DISTINCT sp.id_producto) 
        - COUNT(DISTINCT gpc.id_producto) AS unclassified
FROM workspace.superapp.silver_prices sp
LEFT JOIN workspace.superapp.gold_productos_categorias gpc
    ON sp.id_producto = gpc.id_producto
